# 02 — Pré-processamento

**Dimensão 4 da rúbrica — 15 pontos.**

Cada decisão precisa de justificativa escrita. Decidir *não* criar features é
aceitável, desde que o motivo esteja explícito.

In [18]:
# ============================================================
# 1. CONFIGURAÇÃO DO AMBIENTE
# ============================================================

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns

# Semente fixa para garantir reprodutibilidade
RANDOM_STATE = 42

# Repositório do projeto
REPO_URL = "https://github.com/andrerochadeoliveira/postech-tech-challenge-fase2-template.git"
PROJECT_PATH = Path("/content/postech-tech-challenge-fase2-template")

# Clona o repositório caso ele ainda não esteja disponível
if not PROJECT_PATH.exists():
    !git clone {REPO_URL}
else:
    print("Repositório já disponível nesta sessão.")

# Caminho da base construída no notebook 00
DATA_PATH = PROJECT_PATH / "data" / "processed" / "dataset_base.csv"

# Variável-alvo
TARGET = "target"

pd.set_option("display.max_columns", None)

print("\nDiretório do projeto:")
print(PROJECT_PATH)

print("\nArquivo esperado:")
print(DATA_PATH)

Repositório já disponível nesta sessão.

Diretório do projeto:
/content/postech-tech-challenge-fase2-template

Arquivo esperado:
/content/postech-tech-challenge-fase2-template/data/processed/dataset_base.csv


In [19]:
# ============================================================
# ATUALIZAÇÃO DO REPOSITÓRIO COM AS SUAS ALTERAÇÕES DO GITHUB
# ============================================================

# Entra no diretório do projeto e puxa as atualizações do GitHub
!cd {PROJECT_PATH} && git pull

Already up to date.


In [20]:
# Gera a base analítica necessária para o EDA
!jupyter nbconvert \
    --to notebook \
    --execute \
    "$PROJECT_PATH/notebooks/00_preparacao_dados.ipynb" \
    --output "/tmp/00_preparacao_dados_executado.ipynb"

[NbConvertApp] Converting notebook /content/postech-tech-challenge-fase2-template/notebooks/00_preparacao_dados.ipynb to notebook
0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
[NbConvertApp] Writing 69283 bytes to /tmp/00_preparacao_dados_executado.ipynb


In [21]:
# ============================================================
# VERIFICAÇÃO DA BASE GERADA
# ============================================================

print("Arquivo existe:", DATA_PATH.exists())

if DATA_PATH.exists():
    tamanho_mb = DATA_PATH.stat().st_size / (1024 ** 2)
    print(f"Tamanho do arquivo: {tamanho_mb:.2f} MB")
    print("Caminho:", DATA_PATH)

Arquivo existe: True
Tamanho do arquivo: 1.99 MB
Caminho: /content/postech-tech-challenge-fase2-template/data/processed/dataset_base.csv


In [22]:
# ============================================================
# 2. CARREGAMENTO DA BASE ANALÍTICA
# ============================================================

# Verifica se a base produzida pelo notebook 00 está disponível
if not DATA_PATH.exists():
    raise FileNotFoundError(
        "O arquivo dataset_base.csv não foi encontrado. "
        "Execute primeiro o notebook 00_preparacao_dados.ipynb "
        "para gerar a base analítica."
    )

base = pd.read_csv(DATA_PATH)

print("Base carregada com sucesso.")
print("Dimensão:", base.shape)

qtd_bons = base[base["target"] == 0].shape[0]
qtd_maus = base[base["target"] == 1].shape[0]
perc_bons = (qtd_bons / (qtd_bons + qtd_maus)) * 100
perc_maus = (qtd_maus / (qtd_bons + qtd_maus)) * 100
print(f"Quantidade de bons pagadores: {qtd_bons}")
print(f"Quantidade de maus pagadores: {qtd_maus}")
print(f"Percentual de bons pagadores: {perc_bons:.2f}%")
print(f"Percentual de maus pagadores: {perc_maus:.2f}%")


Base carregada com sucesso.
Dimensão: (16541, 19)
Quantidade de bons pagadores: 16010
Quantidade de maus pagadores: 531
Percentual de bons pagadores: 96.79%
Percentual de maus pagadores: 3.21%


In [23]:
base.head()

,ID,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,DAYS_BIRTH,DAYS_EMPLOYED,FLAG_MOBIL,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,target
0,5008804,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,NaN,2.0,0
1,5008805,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,NaN,2.0,0
2,5008806,M,Y,Y,0,112500.0,Working,Secondary / secondary special,Married,House / apartment,-21474,-1134,1,0,0,0,Security staff,2.0,0
3,5008810,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,-19110,-3051,1,0,1,1,Sales staff,1.0,0
4,5008811,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,-19110,-3051,1,0,1,1,Sales staff,1.0,0


## 1. Dados faltantes

In [24]:
nulos = base.isna().sum()
pd.DataFrame({"nulos": nulos, "pct": (nulos / len(base) * 100).round(2)}).query("nulos > 0")

,nulos,pct
OCCUPATION_TYPE,5090,30.77


**Decisão metodológica — Tratamento dos valores ausentes**

A variável `OCCUPATION_TYPE` apresenta 5.090 valores ausentes, correspondentes a 30,77% dos registros. Considerando a proporção de ausência, optou-se por preservar os clientes e substituir os valores nulos pela categoria `Nao_informado`.

Essa estratégia evita a exclusão de observações e permite que a ausência de informação seja representada explicitamente durante a codificação das variáveis categóricas.

Não foi realizada imputação pela moda, pois isso atribuiria aos clientes uma ocupação não observada nos dados.

In [25]:
base["OCCUPATION_TYPE"] = base["OCCUPATION_TYPE"].fillna("Nao_informado")
base.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16541 entries, 0 to 16540
Data columns (total 19 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   ID                   16541 non-null  int64  
 1   CODE_GENDER          16541 non-null  object 
 2   FLAG_OWN_CAR         16541 non-null  object 
 3   FLAG_OWN_REALTY      16541 non-null  object 
 4   CNT_CHILDREN         16541 non-null  int64  
 5   AMT_INCOME_TOTAL     16541 non-null  float64
 6   NAME_INCOME_TYPE     16541 non-null  object 
 7   NAME_EDUCATION_TYPE  16541 non-null  object 
 8   NAME_FAMILY_STATUS   16541 non-null  object 
 9   NAME_HOUSING_TYPE    16541 non-null  object 
 10  DAYS_BIRTH           16541 non-null  int64  
 11  DAYS_EMPLOYED        16541 non-null  int64  
 12  FLAG_MOBIL           16541 non-null  int64  
 13  FLAG_WORK_PHONE      16541 non-null  int64  
 14  FLAG_PHONE           16541 non-null  int64  
 15  FLAG_EMAIL           16541 non-null 

## 2. Definição da variável alvo

Tratamento realizado no notebook _00_preparacao_dados.ipynb_, atributo target já presente na base

## 3. Transformação de Dados

Nesta etapa, as variáveis originais são transformadas para facilitar sua utilização pelos algoritmos de aprendizado de máquina, preservando as informações relevantes identificadas na análise exploratória.

**Conversão de variáveis temporais**

- `DAYS_BIRTH`: convertido para idade em anos (`Age`), utilizando 365,25 dias por ano.
- `DAYS_EMPLOYED`: transformado em dois atributos:
  - `Employed`: indicador que distingue registros com tempo de emprego convencional daqueles que apresentam o código especial `365243`.
  - `Year_employed`: tempo de emprego convertido para anos. Para registros com o código especial, foi atribuído o valor zero, mantendo-se a distinção por meio do indicador `Employed`.

O código `365243` não representa um período real de emprego e, portanto, não deve ser utilizado diretamente como valor numérico.

**Codificação de variáveis categóricas**

As variáveis `CODE_GENDER`, `FLAG_OWN_CAR` e `FLAG_OWN_REALTY` são convertidas em indicadores binários (0 e 1).

As variáveis `NAME_INCOME_TYPE`, `NAME_EDUCATION_TYPE`, `NAME_FAMILY_STATUS`, `NAME_HOUSING_TYPE` e `OCCUPATION_TYPE` são transformadas por *one-hot encoding*, com exclusão da primeira categoria de cada variável (`drop_first=True`) para reduzir redundâncias.

**Exclusão de atributos originais**

Após as transformações, são removidas as colunas originais que passaram a ser representadas por novos atributos. Também é excluída `FLAG_MOBIL`, por apresentar valor constante na base analisada e não oferecer capacidade de discriminação entre os clientes.

In [26]:
df_n = base.copy()

df_n["Age"] = (df_n["DAYS_BIRTH"] / -365.25)
df_n["Employed"] = (df_n["DAYS_EMPLOYED"] != 365243).astype(int)
df_n["Year_employed"] = np.where(df_n["Employed"] == 0, 0.0, -df_n["DAYS_EMPLOYED"] / 365.25)
df_n["Gender"] = (df_n["CODE_GENDER"] == "M").astype(int)  # 1 = M, 0 = F
df_n["Own_car"] = (df_n["FLAG_OWN_CAR"] == "Y").astype(int)
df_n["Own_realty"] = (df_n["FLAG_OWN_REALTY"] == "Y").astype(int)

multi_cat = [
    "NAME_INCOME_TYPE",
    "NAME_EDUCATION_TYPE",
    "NAME_FAMILY_STATUS",
    "NAME_HOUSING_TYPE",
    "OCCUPATION_TYPE",
]
df_n = pd.get_dummies(df_n, columns=multi_cat, drop_first=True) # One-hot encoding simples eliminando a primeira categoria (evitar redundância)

descartar = ["CODE_GENDER", "FLAG_OWN_CAR", "FLAG_OWN_REALTY", "DAYS_BIRTH", "DAYS_EMPLOYED", "FLAG_MOBIL", "FLAG_MOBIL"]
df_n = df_n.drop(columns=descartar, errors="ignore")

print("Shape:", df_n.shape)
print(df_n.info())

Shape: (16541, 49)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16541 entries, 0 to 16540
Data columns (total 49 columns):
 #   Column                                             Non-Null Count  Dtype  
---  ------                                             --------------  -----  
 0   ID                                                 16541 non-null  int64  
 1   CNT_CHILDREN                                       16541 non-null  int64  
 2   AMT_INCOME_TOTAL                                   16541 non-null  float64
 3   FLAG_WORK_PHONE                                    16541 non-null  int64  
 4   FLAG_PHONE                                         16541 non-null  int64  
 5   FLAG_EMAIL                                         16541 non-null  int64  
 6   CNT_FAM_MEMBERS                                    16541 non-null  float64
 7   target                                             16541 non-null  int64  
 8   Age                                                16541 non-null  

In [27]:
df_n.head()

,ID,CNT_CHILDREN,AMT_INCOME_TOTAL,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,CNT_FAM_MEMBERS,target,Age,Employed,Year_employed,Gender,Own_car,Own_realty,NAME_INCOME_TYPE_Pensioner,NAME_INCOME_TYPE_State servant,NAME_INCOME_TYPE_Student,NAME_INCOME_TYPE_Working,NAME_EDUCATION_TYPE_Higher education,NAME_EDUCATION_TYPE_Incomplete higher,NAME_EDUCATION_TYPE_Lower secondary,NAME_EDUCATION_TYPE_Secondary / secondary special,NAME_FAMILY_STATUS_Married,NAME_FAMILY_STATUS_Separated,NAME_FAMILY_STATUS_Single / not married,NAME_FAMILY_STATUS_Widow,NAME_HOUSING_TYPE_House / apartment,NAME_HOUSING_TYPE_Municipal apartment,NAME_HOUSING_TYPE_Office apartment,NAME_HOUSING_TYPE_Rented apartment,NAME_HOUSING_TYPE_With parents,OCCUPATION_TYPE_Cleaning staff,OCCUPATION_TYPE_Cooking staff,OCCUPATION_TYPE_Core staff,OCCUPATION_TYPE_Drivers,OCCUPATION_TYPE_HR staff,OCCUPATION_TYPE_High skill tech staff,OCCUPATION_TYPE_IT staff,OCCUPATION_TYPE_Laborers,OCCUPATION_TYPE_Low-skill Laborers,OCCUPATION_TYPE_Managers,OCCUPATION_TYPE_Medicine staff,OCCUPATION_TYPE_Nao_informado,OCCUPATION_TYPE_Private service staff,OCCUPATION_TYPE_Realty agents,OCCUPATION_TYPE_Sales staff,OCCUPATION_TYPE_Secretaries,OCCUPATION_TYPE_Security staff,OCCUPATION_TYPE_Waiters/barmen staff
0,5008804,0,427500.0,1,0,0,2.0,0,32.867899,1,12.435318,1,1,1,False,False,False,True,True,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False
1,5008805,0,427500.0,1,0,0,2.0,0,32.867899,1,12.435318,1,1,1,False,False,False,True,True,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False
2,5008806,0,112500.0,0,0,0,2.0,0,58.792608,1,3.104723,1,1,1,False,False,False,True,False,False,False,True,True,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False
3,5008810,0,270000.0,0,1,1,1.0,0,52.320329,1,8.353183,0,0,1,False,False,False,False,False,False,False,True,False,False,True,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False
4,5008811,0,270000.0,0,1,1,1.0,0,52.320329,1,8.353183,0,0,1,False,False,False,False,False,False,False,True,False,False,True,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False


## 4. Normalização / padronização

**Decisão metodológica:** A normalização ou padronização das variáveis numéricas será realizada na etapa de modelagem, no notebook 03, conforme as características dos algoritmos selecionados.

Essa decisão evita o vazamento de informações (*data leakage*), pois os parâmetros de transformação, como média e desvio-padrão, deverão ser estimados exclusivamente a partir dos dados de treinamento.

Para algoritmos sensíveis à escala, poderá ser utilizado o `StandardScaler` dentro de um `Pipeline` do Scikit-learn. Para modelos baseados em árvores, a padronização poderá ser dispensada.

Dessa forma, o notebook 02 disponibilizará os dados transformados, mantendo a definição da estratégia de escalonamento vinculada ao treinamento e à avaliação dos modelos.

## 5. Feature engineering

Nesta etapa, são construídos novos atributos a partir das variáveis cadastrais existentes, com o objetivo de representar características potencialmente relacionadas ao risco de crédito.

**1. Renda por membro familiar (`Income_per_capta`)**

Calculada pela divisão de `AMT_INCOME_TOTAL` por `CNT_FAM_MEMBERS`. Busca representar a renda declarada em relação ao tamanho da família, oferecendo uma perspectiva complementar à renda total.

**2. Relação entre tempo de emprego e idade (`Ratio_life_employed`)**

Calculada pela divisão de `Year_employed` por `Age`. Busca representar a duração do vínculo profissional registrado em relação à idade do cliente.

Essa variável não corresponde necessariamente à proporção de toda a vida profissional, pois a base não contém o histórico completo de empregos.

**3. Indicador simplificado de posse de bens (`Assets_score`)**

Construído pela soma dos indicadores `Own_car` e `Own_realty`, assumindo valores entre 0 e 2.

Seu objetivo é sintetizar a quantidade de categorias de bens declaradas pelo cliente, sem representar valores patrimoniais ou garantias de crédito.

**Decisão metodológica:** Os três atributos serão mantidos para avaliação durante a modelagem. Sua contribuição preditiva deverá ser verificada empiricamente, sem pressupor que necessariamente melhorarão o desempenho dos modelos.

In [28]:
df_feat_eng = df_n.copy()

# 1. Relação renda por membro familiar (evitando divisão por zero se CNT_FAM_MEMBERS for zero ou nulo)
df_feat_eng["Income_per_capta"] = df_feat_eng["AMT_INCOME_TOTAL"] / df_feat_eng["CNT_FAM_MEMBERS"].replace(0, 1)

# 2. Proporção de vida trabalhada
df_feat_eng["Ratio_life_employed"] = np.where(df_feat_eng["Age"] > 0, df_feat_eng["Year_employed"] / df_feat_eng["Age"], 0.0)

# 3. Faixas etárias
#faixas = [0, 30, 45, 60, np.inf]
#labels = ["Ate 30 anos", "31-45 anos", "46-60 anos", "Mais de 60 anos"]
#df_feat_eng["Age_group"] = pd.cut(df_feat_eng["Age"], bins=faixas, labels=labels)

# Como criamos uma coluna categórica com texto, aplicamos One-Hot Encoding nela também (o pandas já descarta 'Age_group')
#df_feat_eng = pd.get_dummies(df_feat_eng, columns=["Age_group"], drop_first=True)

# 4. Interação de posse de bens (Soma de Own_car e Own_realty)
df_feat_eng["Assets_score"] = df_feat_eng["Own_car"] + df_feat_eng["Own_realty"]

# [OPCIONAL] Remoção de variáveis que foram totalmente representadas pelas novas colunas
# Se fôssemos remover a idade contínua para evitar colinearidade com as faixas etárias:
colunas_para_remover = ["FLAG_OWN_CAR", "FLAG_OWN_REALTY"]
df_feat_eng = df_feat_eng.drop(columns=colunas_para_remover, errors="ignore")

# Reordena as colunas para que a 'target' fique no final de tudo
colunas = [col for col in df_feat_eng.columns if col != "target"] + ["target"]
df_feat_eng = df_feat_eng[colunas]

print("Novas colunas na base:", [c for c in df_feat_eng.columns if c not in df_n.columns])
print("Shape", df_feat_eng.shape)
print(df_feat_eng.info())

Novas colunas na base: ['Income_per_capta', 'Ratio_life_employed', 'Assets_score']
Shape (16541, 52)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16541 entries, 0 to 16540
Data columns (total 52 columns):
 #   Column                                             Non-Null Count  Dtype  
---  ------                                             --------------  -----  
 0   ID                                                 16541 non-null  int64  
 1   CNT_CHILDREN                                       16541 non-null  int64  
 2   AMT_INCOME_TOTAL                                   16541 non-null  float64
 3   FLAG_WORK_PHONE                                    16541 non-null  int64  
 4   FLAG_PHONE                                         16541 non-null  int64  
 5   FLAG_EMAIL                                         16541 non-null  int64  
 6   CNT_FAM_MEMBERS                                    16541 non-null  float64
 7   Age                                                16541 non-null

In [29]:
df_feat_eng.head()

,ID,CNT_CHILDREN,AMT_INCOME_TOTAL,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,CNT_FAM_MEMBERS,Age,Employed,Year_employed,Gender,Own_car,Own_realty,NAME_INCOME_TYPE_Pensioner,NAME_INCOME_TYPE_State servant,NAME_INCOME_TYPE_Student,NAME_INCOME_TYPE_Working,NAME_EDUCATION_TYPE_Higher education,NAME_EDUCATION_TYPE_Incomplete higher,NAME_EDUCATION_TYPE_Lower secondary,NAME_EDUCATION_TYPE_Secondary / secondary special,NAME_FAMILY_STATUS_Married,NAME_FAMILY_STATUS_Separated,NAME_FAMILY_STATUS_Single / not married,NAME_FAMILY_STATUS_Widow,NAME_HOUSING_TYPE_House / apartment,NAME_HOUSING_TYPE_Municipal apartment,NAME_HOUSING_TYPE_Office apartment,NAME_HOUSING_TYPE_Rented apartment,NAME_HOUSING_TYPE_With parents,OCCUPATION_TYPE_Cleaning staff,OCCUPATION_TYPE_Cooking staff,OCCUPATION_TYPE_Core staff,OCCUPATION_TYPE_Drivers,OCCUPATION_TYPE_HR staff,OCCUPATION_TYPE_High skill tech staff,OCCUPATION_TYPE_IT staff,OCCUPATION_TYPE_Laborers,OCCUPATION_TYPE_Low-skill Laborers,OCCUPATION_TYPE_Managers,OCCUPATION_TYPE_Medicine staff,OCCUPATION_TYPE_Nao_informado,OCCUPATION_TYPE_Private service staff,OCCUPATION_TYPE_Realty agents,OCCUPATION_TYPE_Sales staff,OCCUPATION_TYPE_Secretaries,OCCUPATION_TYPE_Security staff,OCCUPATION_TYPE_Waiters/barmen staff,Income_per_capta,Ratio_life_employed,Assets_score,target
0,5008804,0,427500.0,1,0,0,2.0,32.867899,1,12.435318,1,1,1,False,False,False,True,True,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,213750.0,0.378342,2,0
1,5008805,0,427500.0,1,0,0,2.0,32.867899,1,12.435318,1,1,1,False,False,False,True,True,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,213750.0,0.378342,2,0
2,5008806,0,112500.0,0,0,0,2.0,58.792608,1,3.104723,1,1,1,False,False,False,True,False,False,False,True,True,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,56250.0,0.052808,2,0
3,5008810,0,270000.0,0,1,1,1.0,52.320329,1,8.353183,0,0,1,False,False,False,False,False,False,False,True,False,False,True,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,270000.0,0.159655,1,0
4,5008811,0,270000.0,0,1,1,1.0,52.320329,1,8.353183,0,0,1,False,False,False,False,False,False,False,True,False,False,True,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,270000.0,0.159655,1,0


## 6. Tratando "gêmeos"

Vamos criar agrupamentos para colocar os gêmes e mesmo grupo

In [30]:
df_g = df_feat_eng.copy()
colunas_cadastro = [c for c in df_g.columns if c not in ["ID", "target"]]
df_g["grupo_perfil"] = pd.util.hash_pandas_object(df_g[colunas_cadastro].astype(str), index=False).values
print("Quantidade de perfis distintos:", df_g["grupo_perfil"].nunique())

Quantidade de perfis distintos: 6923


In [31]:
# Agrupa por perfil e calcula o número de valores únicos de target em cada um
targets_por_grupo = df_g.groupby("grupo_perfil")["target"].nunique()

# Total de grupos distintos
total_grupos = len(targets_por_grupo)

# Grupos homogêneos: número de targets únicos é igual a 1
grupos_homogeneos = (targets_por_grupo == 1).sum()

# Grupos conflitantes: número de targets únicos é maior que 1
grupos_conflitantes = (targets_por_grupo > 1).sum()

# Cálculo das proporções percentuais
pct_homogeneos = (grupos_homogeneos / total_grupos) * 100
pct_conflitantes = (grupos_conflitantes / total_grupos) * 100

print(f"Total de grupos de perfis cadastrais: {total_grupos}\n")
print(f"Grupos HOMOGÊNEOS (mesmo target em todos os registros): {grupos_homogeneos} ({pct_homogeneos:.2f}%)")
print(f"Grupos CONFLITANTES (pelo menos um target diferente): {grupos_conflitantes} ({pct_conflitantes:.2f}%)")

Total de grupos de perfis cadastrais: 6923

Grupos HOMOGÊNEOS (mesmo target em todos os registros): 6649 (96.04%)
Grupos CONFLITANTES (pelo menos um target diferente): 274 (3.96%)


### Decisão metodológica — Consolidação dos perfis cadastrais

Foram identificados **6.923 perfis cadastrais distintos**, dos quais **274 apresentam classificações conflitantes da variável-alvo**.

Por decisão metodológica do projeto, adotou-se uma regra conservadora: quando um perfil cadastral apresenta pelo menos um registro com `target = 1`, todos os registros pertencentes ao mesmo perfil recebem essa classificação.

Após a consolidação, a quantidade de registros com `target = 1` passou de **531 (3,21%) para 1.308 (7,91%)**, mantendo-se o total de 16.541 registros.

Essa transformação representa uma redefinição operacional da variável-alvo para fins de modelagem. Portanto, os registros reclassificados não devem ser interpretados como eventos de inadimplência individualmente observados.

A regra será mantida nas etapas posteriores, devendo ser considerada na interpretação dos resultados e das métricas de desempenho.

In [32]:
# ============================================================
# CONSOLIDAÇÃO DE CONFLITOS (GÊMEOS) E REMOÇÃO DE COLUNAS
# ============================================================

# Identificando os IDs dos grupos que possuem conflito de target
targets_por_grupo = df_g.groupby("grupo_perfil")["target"].nunique()
grupos_conflitantes_ids = targets_por_grupo[targets_por_grupo > 1].index

print(f"Quantidade de grupos conflitantes mapeados: {len(grupos_conflitantes_ids)}")
print(f"Distribuição original do Target:\n{df_g['target'].value_counts()}\n")

# Consolidando os perfis: se o perfil for conflitante, todo o grupo recebe target = 1
df_g.loc[df_g["grupo_perfil"].isin(grupos_conflitantes_ids), "target"] = 1

print(f"Distribuição do Target após consolidação dos conflitantes (atribuído 1):\n{df_g['target'].value_counts()}\n")

# Removendo a coluna 'Employed' conforme solicitado
#if "Employed" in df_g.columns:
#    df_g = df_g.drop(columns=["Employed"])
#    print("Coluna 'Employed' removida com sucesso!")

# Atualizando a lista de colunas para garantir que a 'target' continue no final de tudo
#colunas = [col for col in df_g.columns if col != "target"] + ["target"]
#df_g = df_g[colunas]

Quantidade de grupos conflitantes mapeados: 274
Distribuição original do Target:
target
0    16010
1      531
Name: count, dtype: int64

Distribuição do Target após consolidação dos conflitantes (atribuído 1):
target
0    15233
1     1308
Name: count, dtype: int64



In [33]:
df_g.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16541 entries, 0 to 16540
Data columns (total 53 columns):
 #   Column                                             Non-Null Count  Dtype  
---  ------                                             --------------  -----  
 0   ID                                                 16541 non-null  int64  
 1   CNT_CHILDREN                                       16541 non-null  int64  
 2   AMT_INCOME_TOTAL                                   16541 non-null  float64
 3   FLAG_WORK_PHONE                                    16541 non-null  int64  
 4   FLAG_PHONE                                         16541 non-null  int64  
 5   FLAG_EMAIL                                         16541 non-null  int64  
 6   CNT_FAM_MEMBERS                                    16541 non-null  float64
 7   Age                                                16541 non-null  float64
 8   Employed                                           16541 non-null  int64  
 9   Year_e

## 7. Exportação do dataset pré-processado

Após o tratamento dos valores ausentes, a transformação das variáveis categóricas e temporais, a criação de novos atributos e a consolidação dos perfis cadastrais, a base final é exportada para o arquivo `data/processed/dataset_preprocessed.csv`.

O conjunto resultante contém **16.541 registros e 53 colunas**, preservando todos os clientes da base original.

A distribuição da variável-alvo reflete a regra de consolidação dos perfis cadastrais definida pelo grupo:

- `target = 0`: 15.233 registros.
- `target = 1`: 1.308 registros.

As colunas `ID` e `grupo_perfil` são preservadas para identificação e controle dos registros, mas não deverão ser utilizadas como variáveis preditoras.

A normalização ou padronização, quando necessária, será realizada na etapa de modelagem, utilizando exclusivamente os dados de treinamento para estimar os parâmetros de transformação.

In [34]:
# ============================================================
# EXPORTAÇÃO DA BASE ANALÍTICA
# ============================================================

# Pasta destinada às bases processadas
PROCESSED_PATH = PROJECT_PATH / "data" / "processed"

# Garante que a pasta exista
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

# Caminho da base que será utilizada pelo Notebook 03
DATASET_BASE_PATH = PROCESSED_PATH / "dataset_preprocessed.csv"

# Exporta a base final de modelagem
df_g.to_csv(
    DATASET_BASE_PATH,
    index=False
)

print("=" * 60)
print("EXPORTAÇÃO DA BASE ANALÍTICA")
print("=" * 60)

print(f"Arquivo: {DATASET_BASE_PATH}")
print(f"Dimensões: {df_g.shape}")
print(f"Clientes: {df_g['ID'].nunique():,}")

EXPORTAÇÃO DA BASE ANALÍTICA
Arquivo: /content/postech-tech-challenge-fase2-template/data/processed/dataset_preprocessed.csv
Dimensões: (16541, 53)
Clientes: 16,541
